# DATA 501 — Weekly Assignment 9
## SQL III — Structuring & Modifying Databases: Fixing It at the Source

**Name:** *Riley Sapp*

**NetID:** *Riley Sapp*

---

### Scenario

Week nine at **Ride Knox**. Your Assignment 8 pull request went out with the
station ranking on it, and the ops lead forwarded it to the data team. Their
reply came back the same afternoon:

> *"Nice queries. But you're carrying our mess around in every one of them.
> Our billing system is switching to product codes next month, we've got a
> dock expansion landing, a station opening, and finance keeps asking why the
> station report takes eight seconds to load. Stop working around the
> database. Come fix it — and leave us something that refuses the mess next
> time."*

That is this week, in one paragraph. You get **write access**: the verbs that
change rows, the DDL that changes the rules, the indexes that make the whole
thing fast, and the `sqlite3` connection that finally lets Python read from a
database instead of a CSV.

### One heads-up: house rules

The data team's requests are deliberately **not** the ones we worked in class:
**different stations, different columns, different target values, different
thresholds, different date windows, and a different table to design.** The
*skills* are identical to the slides, but **statements pasted in unchanged
from the class demo will change the wrong rows or return the wrong numbers.**
Read each task and use the values it actually asks for.

### Setup

- You need **`ride_knox.db`** — the same file from Assignments 7–8. If you
  deleted it, download it again from Canvas. Put it **in the same folder as
  this notebook**.
- Everything else runs from this notebook. Part 0 gives you a provided setup
  cell; run it before anything else.
- **The golden rule still holds.** You are about to run destructive
  statements, so you will never touch `ride_knox.db` itself. Part 0 makes a
  working copy called `ride_knox_work.db`, and **rebuilds it from scratch
  every time you run the notebook.** That is what makes Restart & Run All
  work on an assignment that deletes rows.

### Instructions

- Work the parts **in order** — later parts depend on rows and tables that
  earlier parts create.
- Complete every cell marked **`# TODO`** and answer every **Q:** prompt in
  the markdown cells.
- **Write your SQL as SQL.** Keywords uppercase, one clause per line, text in
  single quotes, table aliases once two tables are in play. A statement that
  would embarrass you in a pull request loses points even if it runs.
- **Predict the row count before you run any `UPDATE` or `DELETE`.** Several
  tasks ask you to write the prediction down first. The row count a statement
  reports is your receipt.
- Run each cell as you go (**Shift+Enter**).
- Before submitting: **Kernel → Restart & Run All**. The notebook must
  rebuild the working database from `ride_knox.db` and run top to bottom with
  no errors.
- Submit the `.ipynb` file on Canvas.

### Anchors you can check yourself against

| After… | You should have |
|---|---|
| Part 0 (fresh copy) | `trips` = **250,000** rows; `stations` = **24** rows; **6** spellings of `rider_type` |
| Part 2 | exactly **2** rider-type values, **158,542** + **91,458** |
| Part 3 | `stations` = **25** rows; `trips` = **249,253** rows |
| Part 6 | your summary table has **25** rows, one of them with **0** trips |

If your numbers can't be reconciled with those, something is off — find it
before you keep going.

### Grading (100 points)

| Part | Points |
|---|---|
| 0. Setup & the working copy | 5 |
| 1. Transactions & the safety habit | 8 |
| 2. `UPDATE`: fixing it at the source | 12 |
| 3. `INSERT` & `DELETE` | 12 |
| 4. `CREATE TABLE` & constraints | 14 |
| 5. Foreign keys: declared vs. enforced | 8 |
| 6. Schema design & migration | 12 |
| 7. Indexes & query plans | 14 |
| 8. Connecting Python to SQL | 10 |
| Reflection + AI disclosure | 5 |
| Notebook runs top-to-bottom; SQL is readable and commented | *(required — up to −10 if not)* |
| 9. Challenge (optional) | *+5 extra credit* |

---
## Part 0 — Setup & the Working Copy (5 pts)

**Run the next cell without modifying it.** It does five things worth
understanding, because you would have to do all five yourself in a real job:

1. Deletes any previous working copy and makes a **fresh** one from
   `ride_knox.db`, so this notebook is reproducible no matter how many times
   you run it.
2. Opens a connection with `sqlite3.connect()` (Module 9) — the same call
   from the slides.
3. Sets `isolation_level = None` so that `BEGIN`, `COMMIT`, and `ROLLBACK`
   behave exactly as you type them, instead of Python managing transactions
   behind your back.
4. Defines `run(sql)` — executes one statement and returns the **rows
   affected**. This is your receipt.
5. Defines `try_run(sql)` for statements you *expect* to be refused,
   `timed(sql)` for Part 7, and `db_size_mb()` for measuring what indexes
   cost.

You will write **SQL**; these helpers just carry it to the database. Use
`pd.read_sql()` directly whenever a task asks you to *look* at rows.

In [1]:
# --- PROVIDED SETUP — run as-is, do not modify ---
import os
import shutil
import sqlite3
import time

import pandas as pd
import matplotlib.pyplot as plt

SOURCE = "ride_knox.db"        # never written to — the golden rule
WORK   = "ride_knox_work.db"   # your sandbox, rebuilt on every run

if os.path.exists(WORK):
    os.remove(WORK)
shutil.copy(SOURCE, WORK)

con = sqlite3.connect(WORK)
con.isolation_level = None     # BEGIN / COMMIT / ROLLBACK behave as written


def run(sql):
    """Execute ONE statement. Returns rows affected (-1 if it changes no rows)."""
    return con.execute(sql).rowcount


def try_run(sql):
    """For statements you EXPECT to be refused: prints the database's
    complaint instead of stopping the notebook."""
    try:
        n = con.execute(sql).rowcount
        return f"ACCEPTED ({n} rows) — is that what you expected?"
    except sqlite3.Error as e:
        return f"REFUSED -> {e}"


def timed(sql, params=(), repeat=5):
    """Run a query a few times; return the fastest run, in milliseconds."""
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        con.execute(sql, params).fetchall()
        best = min(best, time.perf_counter() - t0)
    return round(best * 1000, 1)


def db_size_mb():
    return round(os.path.getsize(WORK) / 1_048_576, 1)


print("fresh working copy:", WORK, "|", db_size_mb(), "MB")

fresh working copy: ride_knox_work.db | 23.2 MB


**0a.** Confirm you have the right database. Use `pd.read_sql()` to run the
rider-type count from the Module 8 slides — `rider_type`, `COUNT(*)`,
`GROUP BY rider_type` — exactly as it appears there. *(This is the one place
in this assignment where a class query is used unchanged; it is a baseline
you will destroy in Part 2.)* You should get **six** rows.

**0b.** Print the row count of `trips` and of `stations`, and print
`db_size_mb()`. Write the three numbers down somewhere — Part 7 compares
against the size.

**Q0c:** You are about to run `UPDATE` and `DELETE` statements for the first
time in this course. Name the **two** distinct protections the provided cell
gives you, and say what each one would save you from. (One of them is a line
of Python; the other is a habit the line enforces.)

In [2]:
# TODO 0a: the six rider_type spellings (class query, unchanged)
trips = pd.read_sql("""
SELECT rider_type, COUNT(*) AS trips
FROM trips
GROUP BY rider_type
ORDER BY trips DESC
""", con)
stations = pd.read_excel("stations.xlsx")

# TODO 0b: row counts for trips and stations, plus db_size_mb()
print(len(trips))
# Prints 6
print(len(stations))
# Prints 24

6
24


**Q0c:** *UPDATE and DELETE change the rows themselves. Firstly, the row count acts as a receipt, which guarantees that you are changing the correct rows and lets you check back to see what has changed. Secondly, it enforces the habit of checking the row count as you go, so that if you make a mistake, you are able to spot it quickly.*

---
## Part 1 — Transactions & the Safety Habit (8 pts)

Before you change anything for real, prove to yourself that the undo works.

**1a.** The fleet team asks — *hypothetically* — what the database would look
like with the electric bikes taken out. Do it **inside a transaction you
never commit**:

- `BEGIN TRANSACTION`
- a `DELETE` that removes every trip taken on an **electric** bike
- print the rows affected
- `ROLLBACK`
- print `SELECT COUNT(*) FROM trips` to prove the rows came back

*(Class ran the missing-`WHERE` demo on `stations` with an `UPDATE`. Same
lesson, different table, different verb, and this time the `WHERE` is
deliberate.)*

**1b.** Now the habit the module actually wants. Part 2 will change the dock
counts at **The Hill – Ayres Hall (`S07`)** and **Cumberland Ave & 17th St
(`S11`)**. Write the `SELECT` **first**: return `station_id`, `station_name`,
and `docks` for exactly those two stations, using the same `WHERE` condition
you will reuse in Part 2. Confirm you get 2 rows.

**Q1c:** Your 1a `DELETE` reported a row count. Suppose it had reported
**250,000** instead. Describe exactly what you would type next, and explain
why that recovery would be impossible if you had typed `COMMIT;` first.

**Q1d:** Module 5 gave you Git; this module gives you transactions. A
classmate says *"I don't need `BEGIN` — my database file is in my repo, so I
can always `git checkout` the old version."* Give the one-sentence reason the
module would reject that.

In [3]:
# TODO 1a: BEGIN, delete all electric-bike trips, print rows affected, ROLLBACK, re-count
con.execute("BEGIN TRANSACTION;")
db_no_electric = con.execute("""
    DELETE FROM trips
    WHERE LOWER(bike_type) = 'electric';
""")
print("Number of rows affected is", db_no_electric.rowcount)
con.rollback()
print(pd.read_sql(
    "SELECT COUNT(*) AS total_trips FROM trips",
    con
))

# TODO 1b: the SELECT-first check for S07 and S11
trips = pd.read_sql("""
SELECT station_id, station_name, docks AS trips
FROM stations
WHERE station_id IN ('S07', 'S11');
""", con)
print(len(trips))

Number of rows affected is 95083
   total_trips
0       250000
2


**Q1c:** *If 1a reported 250,000, that would show that the DELETE accidentally deleted every single row. I would ROLLBACK the query before committing it, as the data would be unrecoverable if I had committed it before checking.*

**Q1d:** *Git protects your code, transactions protect your data--the database would be changed, even if Git had backed up the code that he had written.*

---
## Part 2 — `UPDATE`: Fixing It at the Source (12 pts)

**2a.** The dock expansion. `S07` and `S11` are each getting **4 additional
docks**. Write one `UPDATE` that expresses this **as a delta** — set `docks`
from its own current value rather than typing the new totals — using the
`WHERE` condition you tested in 1b. Print the rows affected, then re-run your
1b `SELECT` to show the new counts.

*(Class set two different stations to an absolute value of 32. Here it is two
other stations and the `docks = docks + …` form.)*

**2b.** Billing's product codes. The six spellings of `rider_type` are being
replaced by two **product codes**, not by lowercase words:

| Every spelling of… | becomes |
|---|---|
| member | `annual` |
| casual | `single` |

Write **two** `UPDATE` statements — one per code — and print the rows
affected by each. Your `WHERE` clauses must catch **all** the spellings of
each category, not just the lowercase one (Module 7 taught you the move).

**2c.** Verify. Re-run your 0a query. You should now see exactly **two** rows,
`annual` with **158,542** and `single` with **91,458**. If you see three or
more, your `WHERE` clauses missed a spelling — fix them before continuing.

**Q2d:** Add your two 2b row counts together and compare the total with the
250,000 rows in `trips`. What does the comparison prove, and what would a
mismatch in either direction have told you?

**Q2e:** In 2a you wrote the change as a delta rather than as absolute
values. Give one situation in which the delta form is **safer**, and one in
which it is **more dangerous** — the second one is the reason the module
insists on checking the row count.

**Q2f:** The module lists three costs of an `UPDATE` like 2b. Name the cost
that applies most sharply here — where the *original* six spellings are now
unrecoverable — and describe one thing Ride Knox could have kept that would
preserve the evidence without leaving the mess in `rider_type`.

In [4]:
# TODO 2a: add 4 docks to S07 and S11 (delta form), print rows affected, then re-check
updated_docks = con.execute("""
UPDATE stations
SET docks = docks + 4
WHERE station_id IN ('S07', 'S11');
""")

# TODO 2b: two UPDATEs -> 'annual' and 'single'; print rows affected for each
trips_member = con.execute("""
UPDATE trips
SET rider_type = 'annual'
WHERE LOWER(rider_type) = 'member';
""")

trips_casual = con.execute("""
UPDATE trips
SET rider_type = 'single'
WHERE LOWER(rider_type) = 'casual';
""")

# TODO 2c: verify — two rows, 158,542 and 91,458
print(trips)

  station_id              station_name  trips
0        S07     The Hill - Ayres Hall     12
1        S11  Cumberland Ave & 17th St     16


**Q2d:** *The 2b row counts equals 250,000. If the row count was smaller than 250,000, it could be assumed that the WHERE LOWER(rider_type) did not catch all possible spellings. If the row count was larger than 250,000, it could be assumed that something the rows being counted were not just the trips.*

**Q2e:** *The delta form is safer when you safely know what the row count is; if you did a row count recently, then using the delta form ensures that you do not accidentally mistype what you mean to, as well as ensuring that it remains updated if the row count changes. However, this can be dangerous because if you do not know the row count, it does not give you any indication that the inputs in the equation changed, so the data can change without you realizing it.*

**Q2f:** *The cost that most aptly applies is that the original row data is unrecoverable--Ride Knox could instead make the change in pandas through code rather than changing the data itself.*

---
## Part 3 — `INSERT` & `DELETE` (12 pts)

**3a.** A new station opens. Insert it into `stations`, **naming your
columns**:

| station_id | station_name | neighborhood | latitude | longitude | docks | year_installed |
|---|---|---|---|---|---|---|
| `S26` | Cherokee Blvd Greenway | Sequoyah Hills | 35.9412 | −83.9702 | 14 | 2026 |

Then confirm `stations` has **25** rows.

**3b.** Prove the primary key is doing its job. Use `try_run()` to attempt the
**same** insert a second time. Paste what comes back and note which
constraint refused it.

**3c.** The impossible trips. Module 8's average duration was poisoned by
trips that ended before they began. Get rid of them **at the source**:

- First, a `SELECT COUNT(*)` that counts trips where `end_time` is not after
  `start_time`. **Write your prediction in the code comment before you run
  it.**
- Then the matching `DELETE`, printing the rows affected.
- Then a re-count of `trips`.

*(Class deleted the 263 test-station trips. This is a different predicate,
a different count, and a different reason.)*

**Q3d:** Your 3c `DELETE` removed rows permanently. The module offers an
alternative that Module 3 also chose — one that keeps the rows but makes them
easy to exclude. Name it, write the two statements you would have run
instead (in prose or SQL, your choice), and say which approach you would
defend to the data team and why.

**Q3e:** The 263 trips that start at the test station `S99` are still sitting
in `trips` — this assignment never deleted them. Give one *analytical*
consequence of leaving them there and one *design* reason a database person
might prefer to leave them alone for now.

In [5]:
# TODO 3a: insert station S26, then confirm 25 rows
new_station = con.execute("""
INSERT INTO stations (station_id, station_name, neighborhood, latitude, longitude, docks, year_installed)
VALUES ('S26', 'Cherokee Blvd Greenway','Sequoyah Hills', 35.9412, -83.9702, 14, 2026);
""")
new_stations = pd.read_sql("""
SELECT COUNT(*) AS stations
FROM stations;
""", con)
print(new_stations)

# TODO 3b: try the same INSERT again with try_run() and read the refusal
try_run("""
    INSERT INTO stations (station_id, station_name, neighborhood, latitude, longitude, docks, year_installed)
VALUES ('S26', 'Cherokee Blvd Greenway','Sequoyah Hills', 35.9412, -83.9702, 14, 2026);
""")
## Error message: 'REFUSED -> UNIQUE constraint failed: stations.station_id'

# TODO 3c: prediction -> 747 rows
#          count them, delete them, re-count trips
impossible = pd.read_sql("""
SELECT COUNT(*) AS impossible_trips
FROM trips
WHERE end_time <= start_time;
""", con)

impossible_delete = con.execute("""
DELETE FROM trips
WHERE end_time <= start_time;
""")
print(impossible_delete.rowcount)

cleaned_trips = pd.read_sql("""
SELECT COUNT(*) AS cleaned_trips
FROM trips;
""", con)
print(cleaned_trips.iloc[0, 0])

   stations
0        25
747
249253


**Q3d:** *Instead of deleting the data, you could instead use filtering, like using SELECT and WHERE to only retrieve the relevant data and then running queries off of that variable. I would recommend the team use the filtering one rather than DML because it allows for colleagues to continue work as expected rather than having to ensure that all stakeholders are aware of the change in data--it also allows there to be a paper trail, in case someone needs to check the historic data in the future.*

**Q3e:** *Leaving the test stations in the data can cause problems because one must remember to filter it out when analyzing the data, if S99 responses would cause a misrepresentation. However, it is beneficial in case a person who is analyzing the data in the future has any specific questions about these test stations (for example, if they wanted to know if the tests were run most frequently at a certain time of day).*

---
## Part 4 — `CREATE TABLE` & Constraints (14 pts)

Part 2 changed two dock counts, and the database now has no memory that they
were ever anything else — the exact problem the slides raised when Hodges
Library's pressure ranking moved. Fix it properly: give Ride Knox a table
that **records** dock changes instead of overwriting them.

**4a.** Create a table named **`dock_changes`** with these columns and rules.
Every rule below must appear as a constraint in your `CREATE TABLE` — do not
enforce any of them in Python.

| Column | Type | Rules |
|---|---|---|
| `change_id` | INTEGER | primary key |
| `station_id` | TEXT | required; must refer to a real row in `stations` |
| `effective_date` | TEXT | required |
| `docks_before` | INTEGER | required; must be greater than 0 |
| `docks_after` | INTEGER | required; must be greater than 0 |
| `reason` | TEXT | required; defaults to `'ops request'` when not supplied |

Plus one **table-level** rule: a row where `docks_after` equals
`docks_before` records no change at all and must be refused.

*(Class designed `trips_v2` for trip rows. Same constraint vocabulary,
entirely different table — and this one solves a problem the slides only
posed.)*

**4b.** Record the real change. Insert **one** valid row documenting the
`S11` expansion you made in Part 2: from 16 docks to 20, effective
`'2026-07-01'`. Leave `reason` **out of your column list** so the default
fills it, then `SELECT *` from the table to show what got stored.

**4c.** Make the schema refuse four different things. Using `try_run()`, run
one statement per line below and paste what each returns:

1. a change for station `'S40'` — a station that does not exist
2. a change from 12 docks to 12 docks
3. a change with `docks_after` of 0
4. a change with no `effective_date` at all

**Q4d:** Three of your four attempts in 4c were refused and **one was
accepted.** Say which one got through and give your best explanation. (You
are not expected to be certain yet — Part 5 confirms it. Write down what you
think is happening.)

**Q4e:** `dock_changes` records a `docks_before` and a `docks_after`, but
`stations.docks` still holds only the current number. Explain in 2–3
sentences how having *both* would have let you compute a defensible
trips-per-dock figure for 2025 after Part 2's expansion — and name the column
you would have to read to do it.

**Q4f:** You could have written a Python `if` statement to reject a
zero-dock row before inserting it. Give the module's argument for putting
that rule in the schema instead, in terms of *who* the rule protects.

In [6]:
# TODO 4a: CREATE TABLE dock_changes with all six constraints
new_table = con.execute("""
CREATE TABLE dock_changes (
change_id INTEGER PRIMARY KEY,
station_id TEXT NOT NULL REFERENCES stations(station_id),
effective_date TEXT,
docks_before INTEGER NOT NULL CHECK (docks_before > 0),
docks_after INTEGER NOT NULL CHECK (docks_after > 0),
reason TEXT NOT NULL DEFAULT 'ops request',
CHECK (docks_after != docks_before)
);
""")

# TODO 4b: insert the S11 change (16 -> 20, 2026-07-01), let `reason` default, then SELECT *
new_row = con.execute("""
INSERT INTO dock_changes (station_id, effective_date, docks_before, docks_after)
VALUES ('S11', '2026-07-01', 16, 20);
""")
changes = pd.read_sql("""
SELECT * FROM dock_changes;
""", con)
print(changes)

# TODO 4c: four try_run() calls — bad station, no-op change, zero docks, missing date
try_run("""
INSERT INTO dock_changes (station_id)
VALUES ('S40')
""")
try_run("""
INSERT INTO dock_changes (docks_before)
VALUES (docks_after)
""")
try_run("""
INSERT INTO dock_changes (docks_after)
VALUES (0)
""")
try_run("""
INSERT INTO dock_changes (effective_date)
VALUES (NULL)
""")

# 'REFUSED -> NOT NULL constraint failed: dock_changes.docks_before'
# 'REFUSED -> no such column: docks_after'
# 'REFUSED -> NOT NULL constraint failed: dock_changes.station_id'

   change_id station_id effective_date  docks_before  docks_after       reason
0          1        S11     2026-07-01            16           20  ops request


'REFUSED -> NOT NULL constraint failed: dock_changes.station_id'

**Q4d:** *The first statement was not refused; this is likely because the station_id was not constrained as not being able to be S40, whereas all the other try_run queries were constrained specifically when creating dock_changes.*

**Q4e:** *Having both a docks_before and a docks_after would let you compute a defensible trips-per-dock figure, as it would allow you to analyze the change that the expansion made. This could be done by making a column called change_in_docks, which would be the difference between docks_before and docks_after.*

**Q4f:** *A schema constraint fixes the cause, not just the symptom: in the future, if a colleague adds data, then the schema constraint is applied to that data, while a Python IF statement is not applied to everyone. It protects everyone rather than just the Python user.*

---
## Part 5 — Foreign Keys: Declared vs. Enforced (8 pts)

**5a.** Ask the database whether it is enforcing foreign keys at all:
`PRAGMA foreign_keys;`. Read the answer, then turn enforcement **on**.

**5b.** Clean up after 4c and prove the fix. Delete the row that referenced
the non-existent station `'S40'`, then use `try_run()` to attempt that same
insert again. It should be refused this time. Paste both results.

**5c.** Audit what was already inside. Run `PRAGMA foreign_key_check;`
through `pd.read_sql()` and report **how many rows** come back and which
table they are in. *(Hint: it is not `dock_changes`.)*

**Q5d:** Your 5c audit found violations even though enforcement is now on.
Explain why turning the pragma on did not fix them, and state the general
rule this illustrates about when a constraint does its work.

**Q5e:** `PRAGMA foreign_keys` is a **per-connection** setting. Describe the
concrete failure this creates for a team: two analysts connect to the same
database file on the same afternoon, and only one of them ran the pragma.
What can each of them do that the other cannot?

In [7]:
# TODO 5a: read PRAGMA foreign_keys, then turn it on
foreign = con.execute("""
PRAGMA foreign_key_check;
""")
print(foreign.fetchone())
con.execute("PRAGMA foreign_keys = ON;")
print(foreign.fetchone())

# TODO 5b: delete the S40 row, then retry the insert with try_run()
try_run("""
INSERT INTO dock_changes (station_id)
VALUES ('S40')
""")

# TODO 5c: PRAGMA foreign_key_check through pd.read_sql — how many rows, which table?
read_foreign = pd.read_sql("""
PRAGMA foreign_key_check;
""", con)
print(read_foreign)
# 263 rows are returned; they are in the original stations table.

('trips', 1076, 'stations', 1)
('trips', 1306, 'stations', 1)
     table   rowid    parent  fkid
0    trips    1076  stations     1
1    trips    1306  stations     1
2    trips    2952  stations     1
3    trips    4168  stations     1
4    trips    4309  stations     1
..     ...     ...       ...   ...
258  trips  246965  stations     1
259  trips  247740  stations     1
260  trips  248739  stations     1
261  trips  249412  stations     1
262  trips  249920  stations     1

[263 rows x 4 columns]


**Q5d:** *Constraints do their work only when they are committed to database. The pragma was only run through pd.read_sql, so it did not affect the database.*

**Q5e:** *The analyst who has turned foreign keys on cannot violate constraints, whereas the analyst who has them off can. This creates problems for the teams, as both would receive different outputs when running the same file.*

---
## Part 6 — Schema Design & Migration (12 pts)

*This part combines two things class never combined: a table you design with
constraints (this module) and an aggregate query across two tables (Module
8). The ops lead wants the station report to stop being a query someone
remembers and start being a table anyone can read.*

**6a.** Design and create a table named **`station_summary_2025`** holding
one row per station, with these columns:

| Column | Type | Rules |
|---|---|---|
| `station_id` | TEXT | primary key; must refer to a real row in `stations` |
| `station_name` | TEXT | required |
| `neighborhood` | TEXT | required |
| `docks` | INTEGER | required; greater than 0 |
| `trips` | INTEGER | required; **zero is allowed** |
| `trips_per_dock` | REAL | required |

**6b.** Fill it with a single `INSERT INTO … SELECT` that computes the
summary from `stations` and `trips`. Requirements:

- **every** station must appear, including one that has no trips at all
- `trips` counts trips that **started** at the station
- `trips_per_dock` is `trips` divided by `docks`, rounded to 1 decimal
- the row for the station with no trips must read **0**, not 1

Then `SELECT COUNT(*)` from your new table. You should get **25**.

**6c.** Read the result back with `pd.read_sql()`, sorted by
`trips_per_dock` descending, and display the **top 6** and the **bottom 3**.

**Q6d:** Two of Module 8's details are what make 6b's "zero, not 1" work.
Name both, and say what your table would have contained if you had used
`COUNT(*)` and an inner join instead.

**Q6e:** `trips_per_dock` is **derived** — it can be recomputed at any time
from `trips` and `docks`. Storing it therefore breaks the "one fact, one
place" rule. Name the normal form this relates to, then give one honest
argument *for* storing it anyway and one thing that will go wrong the moment
someone runs another `UPDATE` on `stations.docks`.

**Q6f:** Ride Knox launches a **Day Pass** product in March 2026, and its
trips will arrive with a `rider_type` your Part 2 codes do not cover. You
have two designs available: widen a `CHECK` constraint, or add a lookup table
that `trips` points at with a foreign key. Pick one, and name the specific
thing you would regret about your own choice.

In [8]:
# TODO 6a: CREATE TABLE station_summary_2025
new_table = con.execute("""
CREATE TABLE station_summary_2025 (
station_id TEXT PRIMARY KEY NOT NULL REFERENCES stations(station_id),
station_name TEXT NOT NULL,
neighborhood TEXT NOT NULL,
docks INTEGER NOT NULL CHECK (docks > 0),
trips INTEGER NOT NULL,
trips_per_dock REAL NOT NULL
);
""")

# TODO 6b: INSERT INTO ... SELECT — every station, zero-safe count, rounded ratio; then COUNT(*)
#new_row = con.execute("""
#INSERT INTO station_summary_2025
#SELECT station_id, station_name, neighborhood, docks, trips, trips_per_dock
#LOWER(rider_type), bike_type
#FROM trips
#WHERE start_station_id <> 'S99'
#AND (end_time IS NULL OR end_time > start_time);
#""", con)

# TODO 6c: read it back with pd.read_sql — top 6 and bottom 3 by trips_per_dock
#read_table = pd.read_sql("""
#SELECT station_summary_2025
#ORDER BY trips_per_dock DESC
#LIMIT 6;
#""", con)

DatabaseError: Execution failed on sql '
SELECT station_summary_2025
ORDER BY trips_per_dock DESC
LIMIT 6;
': no such column: station_summary_2025

**Q6d:** *(Double-click and answer here.)*

**Q6e:** *(Double-click and answer here.)*

**Q6f:** *(Double-click and answer here.)*

---
## Part 7 — Indexes & Query Plans (14 pts)

Finance's eight-second report. Measure first, then fix.

**7a.** Two baselines. For each of these two queries, print both the plan
(`EXPLAIN QUERY PLAN …`, read with `pd.read_sql()`) and the time from
`timed()`:

- **A:** count the trips that **ended** at `S11`
- **B:** count the `single`-code trips that started in **October 2025**
  (use the half-open date range from Module 7, not `BETWEEN`)

Both should report `SCAN`.

**7b.** Build two indexes and re-measure:

- one single-column index on the column query **A** filters
- one **composite** index on the two columns query **B** filters, with the
  equality column first

Re-print both plans and both times, and print `db_size_mb()` before and after.

*(Class indexed `start_time` and `(start_station_id, start_time)`. These are
different columns — and query B's index only works because of what you did in
Part 2.)*

**7c.** Show that column order matters. Run `EXPLAIN QUERY PLAN` on a query
that filters on **only the October date range** — no rider-type condition at
all. Report the exact wording of the plan: which index name appears, and
whether the plan says `SEARCH` or `SCAN`.

**7d.** Defeat your own index. Write query **B** a second way, replacing the
half-open date range with `strftime('%m', start_time) = '10'` and keeping the
rider-type condition unchanged. Print the plan, the time, and the count next
to the 7b version.

**Q7e:** Your 7d version returns the **same count** as 7b on this database —
and it is still not the same question. Describe the circumstance under which
the two would disagree, and which one you would trust.

**Q7f:** Read the 7c and 7d plans side by side. In both, your composite index
is *named* — yet neither query got 7b's speed. For each one, say which
column's condition the index could still serve and which it could not, and
state the general rule about composite indexes that 7c demonstrates.

**Q7g:** Report your `db_size_mb()` before and after as a percentage
increase. Then name the *other* cost indexes carry — the one a size
measurement can't show — and say which Ride Knox operation pays it.

In [ ]:
# TODO 7a: plans and times for query A (ended at S11) and query B (October 'single' trips)


# TODO 7b: create the two indexes, re-print both plans, both times, and db_size_mb() before/after


# TODO 7c: plan for the October date range alone — which index, and SEARCH or SCAN?


# TODO 7d: the strftime version of query B — plan, time, and count

**Q7e:** *(Double-click and answer here.)*

**Q7f:** *(Double-click and answer here.)*

**Q7g:** *(Double-click and answer here.)*

---
## Part 8 — Connecting Python to SQL (10 pts)

**8a.** A parameterized query. Write **one** query string with `?`
placeholders that counts trips starting at a given station within a given
date window, and call it through `pd.read_sql(..., params=...)` for
**`S11` in October 2025**. Then call the *same* string again, changing only
`params`, for **`S06` in December 2025**.

*(Class parameterized S06 in July. The point of 8a is the second call: one
query string, two answers, nothing rebuilt.)*

**8b.** Aggregate in SQL, draw in Python. Pull the **top 8 stations by
`trips_per_dock`** out of `station_summary_2025` with `pd.read_sql()`, then
draw a **labeled horizontal bar chart** (Module 4's `barh`, busiest at the
top). Title, axis label, and readable station names are required.

**8c.** Write something back. Build an hourly profile — trips per hour of
day, per rider-type code — with a `GROUP BY` query, then push the resulting
DataFrame into the database as a table called `hourly_profile` using
`to_sql()`. Then run `PRAGMA table_info(hourly_profile)` and
`PRAGMA table_info(station_summary_2025)` and display both.

**Q8d:** Compare the two `PRAGMA table_info` outputs from 8c column by
column. List every difference you can see, and state the module's warning
about `to_sql()` that those differences illustrate.

**Q8e:** A teammate proposes `pd.read_sql("SELECT * FROM trips", con)`
followed by a pandas `groupby` to build 8b's chart. Both approaches produce
the same picture. Give the module's argument against theirs, with a number
from your own work to make it concrete.

**Q8f:** Rewrite 8a's query as an f-string in your head — do **not** run
it — and describe the two distinct failures the module says `?` placeholders
prevent. One is a bug; one is an attack.

In [ ]:
# TODO 8a: one parameterized query string, called twice with different params


# TODO 8b: top 8 by trips_per_dock -> labeled horizontal bar chart


# TODO 8c: hourly profile query -> to_sql("hourly_profile") -> compare both PRAGMA table_info outputs

**Q8d:** *(Double-click and answer here.)*

**Q8e:** *(Double-click and answer here.)*

**Q8f:** *(Double-click and answer here.)*

---
## Reflection + AI Disclosure (5 pts)

**R1.** This week you ran statements that change data permanently. Describe
one moment in this assignment where you were about to run something and
stopped to check first — what you checked, and what you found. If it never
happened, say so honestly and name the task where it *should* have.

**R2.** Parts 2 and 4 offer two different answers to the same problem: fix
the mess with an `UPDATE`, or refuse it with a constraint. In 2–3 sentences,
say which one you would put in front of the Ride Knox data team first, and
why the other one alone would not be enough.

**R3.** **AI disclosure:** describe any use of generative AI tools in this
assignment, or state "No generative AI tools were used." Example: *"I used a
generative AI tool to explain why my composite index was ignored when I
filtered on the second column. All final statements, results, and
conclusions are my own."*

**R1:** *I did not step to check between tasks, but it would have been most helpful to continuously check the row count after each step to ensure that nothing wrong occurred without me knowing.*

**R2:** *I would put the constraint in front of the Ride Knox data team first, because an UPDATE would change the data itself. Because UPDATE is irrecoverable, I would use the less destructive form first, as I want the Ride Knox data team to be able to make the decision on if they wanted to data that the constraint excludes.*

**R3:** *I used generative AI to correct my syntax for part 0a, as I did not use quotation marks, which caused my query to not run correctly. I also used it for part 1a, where I was encountering a syntax error because Jupyter Notebook would not read SQL--I tried to run it as a pd.read_sql() and by running the query within triple quotation marks inside of a variable, but I repeatedly got errors and instead used the syntax it recommended (con.execute and .rowcount instead of len(), which I used for the rest of the module). I also used it for part 3a for how to put values into INSERTed columns, and for part 3c to determine how to get the length of the cleaned_trips (it suggested using .iloc, whereas I had just been trying to do len([])). I also used it in part 4a because I did not know how to add a DEFAULT rule for a table. I used it in part 5a to troubleshoot why I was not getting a result when using print(foreign), where it showed that the correct command would be print(foreign.fetchone()). All final answers are my own.*

---
## Part 9 — Challenge (optional, +5 extra credit)

The data team liked `dock_changes` and wants it to be more than a filing
cabinet.

**9a.** Backfill the table so it holds a second row: the `S07` expansion from
Part 2 (12 docks to 16, effective `'2026-07-01'`), this time supplying an
explicit `reason` of your own. Then write **one** query that joins
`dock_changes` to `stations` and returns, per change: the station name, the
neighborhood, the date, the before and after counts, and the **percentage
increase** in capacity, rounded to 1 decimal.

**9b.** Use the history to do what Part 2 made impossible. Write one query
against `station_summary_2025` and `dock_changes` that returns, for each
station that has a recorded change: its stored `trips_per_dock` (computed on
the **new** dock count) and what that figure **would have been** on the
`docks_before` count. Show both numbers side by side.

**Q9c:** Your 9b output has two defensible pressure figures for the same
station and the same trips. Say which one belongs in a report about **2025
ridership** and which belongs in a report about **2026 capacity planning**,
and give the one-line caption you would put under each so a reader cannot
confuse them.

In [ ]:
# TODO 9a: backfill the S07 change, then join dock_changes to stations with a % increase


# TODO 9b: stored trips_per_dock vs. what it would have been on docks_before

**Q9c:** *(Double-click and answer here.)*

---
## Before You Submit ✅

- [ ] **Kernel → Restart & Run All** — the working copy rebuilds from
      `ride_knox.db` and every cell runs top to bottom with no errors
- [ ] Every `# TODO` is completed and every **Q:** prompt is answered
- [ ] Your name and NetID are at the top
- [ ] Your anchors reconcile: **2** rider-type codes (158,542 / 91,458),
      **25** stations, **249,253** trips, **25** summary rows with one at **0**
- [ ] Every `UPDATE` and `DELETE` prints its **rows affected**, and Part 3's
      prediction is written in the comment
- [ ] All six `dock_changes` rules live in the `CREATE TABLE`, not in Python
- [ ] Part 7 shows a **plan and a time** for every before/after pair
- [ ] Part 8's chart has a title, an axis label, and readable station names
- [ ] Part 8a uses `?` placeholders with `params=` — no f-strings in any query
- [ ] `ride_knox.db` was never written to; all changes went to
      `ride_knox_work.db`
- [ ] Reflection + AI disclosure completed
- [ ] File saved and uploaded to Canvas as `.ipynb`